# 07 — Processing Audit

This notebook audits different preprocessing choices for synthetic BBH strain data.

The goal is to understand how preprocessing affects the physical information available to a model.

We compare several processing presets:

1. raw identity;
2. bandpass only;
3. whitening only;
4. whitening + bandpass current baseline;
5. whitening + bandpass + standardization;
6. alternative lowpass choices.

We first test them on signal-only controlled samples. Later we will repeat the same analysis with noisy signals.

In [ ]:
from pathlib import Path
import sys
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None, marker: str = "src") -> Path:
    """
    Walk upwards from `start` until finding a directory containing `marker`.

    For this project, the repository root should contain:
        src/
        notebooks/
        configs/
        scripts/
    """
    if start is None:
        start = Path.cwd()

    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (candidate / marker).is_dir():
            return candidate

    raise RuntimeError(
        f"Could not find project root containing '{marker}/' starting from {start}"
    )

print(Path.cwd())
PROJECT_ROOT = find_project_root(Path.cwd() / "cbc_pe")
print("PROJECT_ROOT:", PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("src exists:", (PROJECT_ROOT / "src").exists())

In [ ]:
from src.config import SimulationConfig
from src.parameters import CBCParameters
from src.dataset import DatasetBuilder
from src.processing import SignalProcessor

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

## 1. Base simulation configuration

We use the same physical and numerical setup as the current 32 s BBH files.

At this stage the goal is not to generate a large dataset, but to compare how different preprocessing choices transform the same controlled signals.

In [ ]:
config = SimulationConfig(
    simulation_regime="BBH",
    waveform_family="IMR",

    sampling_frequency=4096.0,
    duration=32.0,

    low_frequency_cutoff=30.0,
    waveform_approximant="SEOBNRv4_opt",

    target_network_snr_range=(10.0, 25.0),
    snr_relative_tolerance=0.05,
    snr_on_truncated_signal=True,

    truncation_policy="keep_last_segment",
    required_final_duration=1.0,

    safe_margin_start=0.0,
    safe_margin_end=0.0,

    processing_context_start_samples=1664,
    processing_context_end_samples=1664,
)

print("duration:", config.duration)
print("sampling_frequency:", config.sampling_frequency)
print("length:", config.length)
print("processing_length:", config.processing_length)
print("processing context start [s]:", config.processing_context_start_seconds)
print("processing context end [s]:", config.processing_context_end_seconds)

## 2. Processing presets

We define a small set of preprocessing presets.

The current baseline is:

- PSD whitening;
- highpass at 30 Hz;
- lowpass at 512 Hz;
- FIR order 256;
- whitening max filter duration 0.5 s;
- output cropped back to the final configured duration.

We compare this against simpler and more controlled alternatives.

In [ ]:
PROCESSING_PRESETS = {
    "P0_raw_identity": {
        "whitening_method": "none",
        "apply_highpass": False,
        "apply_lowpass": False,
        "apply_standardization": False,
        "output_mode": "crop_to_config",
    },

    "P1_bandpass_30_512": {
        "whitening_method": "none",
        "apply_highpass": True,
        "apply_lowpass": True,
        "apply_standardization": False,
        "output_mode": "crop_to_config",
        "highpass_frequency": 30.0,
        "lowpass_frequency": 512.0,
        "fir_order": 256,
        "fir_beta": 5.0,
        "remove_corrupted": True,
    },

    "P2_whiten_only": {
        "whitening_method": "psd",
        "apply_highpass": False,
        "apply_lowpass": False,
        "apply_standardization": False,
        "output_mode": "crop_to_config",
        "whitening_low_frequency_cutoff": 30.0,
        "whitening_max_filter_duration": 0.5,
        "whitening_trunc_method": "hann",
        "remove_corrupted": True,
    },

    "P3_whiten_bandpass_30_512_current": {
        "whitening_method": "psd",
        "apply_highpass": True,
        "apply_lowpass": True,
        "apply_standardization": False,
        "output_mode": "crop_to_config",
        "whitening_low_frequency_cutoff": 30.0,
        "whitening_max_filter_duration": 0.5,
        "whitening_trunc_method": "hann",
        "highpass_frequency": 30.0,
        "lowpass_frequency": 512.0,
        "fir_order": 256,
        "fir_beta": 5.0,
        "remove_corrupted": True,
    },

    "P4_whiten_bandpass_30_512_standardized": {
        "whitening_method": "psd",
        "apply_highpass": True,
        "apply_lowpass": True,
        "apply_standardization": True,
        "output_mode": "crop_to_config",
        "whitening_low_frequency_cutoff": 30.0,
        "whitening_max_filter_duration": 0.5,
        "whitening_trunc_method": "hann",
        "highpass_frequency": 30.0,
        "lowpass_frequency": 512.0,
        "fir_order": 256,
        "fir_beta": 5.0,
        "remove_corrupted": True,
    },

    "P5_whiten_bandpass_30_1024": {
        "whitening_method": "psd",
        "apply_highpass": True,
        "apply_lowpass": True,
        "apply_standardization": False,
        "output_mode": "crop_to_config",
        "whitening_low_frequency_cutoff": 30.0,
        "whitening_max_filter_duration": 0.5,
        "whitening_trunc_method": "hann",
        "highpass_frequency": 30.0,
        "lowpass_frequency": 1024.0,
        "fir_order": 256,
        "fir_beta": 5.0,
        "remove_corrupted": True,
    },
}

In [ ]:
preset_rows = []

for name, kwargs in PROCESSING_PRESETS.items():
    processor = SignalProcessor(
        config=config,
        **kwargs,
    )

    meta = processor.metadata()
    row = {
        "preset": name,
        "processing_preset_name": meta["processing_preset"],
        "whitening_method": meta["whitening_method"],
        "apply_highpass": meta["apply_highpass"],
        "apply_lowpass": meta["apply_lowpass"],
        "apply_standardization": meta["apply_standardization"],
        "highpass_frequency": meta["highpass_frequency"],
        "lowpass_frequency": meta["lowpass_frequency"],
        "corrupted_margin_seconds_per_side": meta["corrupted_margin_seconds_per_side"],
        "recommended_safe_margin_start": meta["recommended_safe_margin_start"],
        "recommended_safe_margin_end": meta["recommended_safe_margin_end"],
        "usable_duration_after_processing_margins": meta["usable_duration_after_processing_margins"],
        "output_duration": meta["output_duration"],
        "processing_input_duration": meta["processing_input_duration"],
    }
    preset_rows.append(row)

preset_df = pd.DataFrame(preset_rows)
preset_df

## 3. Controlled signal-only samples

We first generate one clean signal per mass group with fixed extrinsic parameters.

This removes nuisance variability and allows us to inspect whether preprocessing preserves the expected mass-dependent morphology.

In [ ]:
def make_fixed_params(mass, distance=1000.0):
    return CBCParameters(
        mass_1=float(mass),
        mass_2=float(mass),
        distance=float(distance),
        inclination=0.7,
        ra=1.0,
        dec=0.5,
        spin_1z=0.0,
        spin_2z=0.0,
        polarization_angle=0.0,
    )


mass_groups = [20.0, 40.0, 60.0, 80.0]

fixed_params = [
    make_fixed_params(mass, distance=1000.0)
    for mass in mass_groups
]

fixed_labels = np.asarray(mass_groups)

In [ ]:
detector_names = ["H1", "L1", "V1"]

# Use current baseline processor in builder only to construct the object.
# We will apply the processing presets manually later.
builder = DatasetBuilder.from_config(
    config=config,
    detector_names=detector_names,
    signal_processor_kwargs=PROCESSING_PRESETS["P3_whiten_bandpass_30_512_current"],
    label_transformer_kwargs={},
    parameter_sampler_kwargs={
        "regime": "BBH",
        "fixed": {},
    },
    rng=np.random.default_rng(1234),
)

In [ ]:
def build_signal_only_segments(
    builder,
    params_list,
    labels,
    geocentric_coalescence_time=1126259462.0,
    placement_policy="end_aligned",
):
    X_list = []
    rows = []

    for i, (params, label) in enumerate(zip(params_list, labels)):
        network = builder._build_projected_signal_network(
            params=params,
            geocentric_coalescence_time=geocentric_coalescence_time,
            placement_policy=placement_policy,
        )

        X_i = np.stack(
            [np.asarray(network.signal_segments[det]) for det in builder.detector_names],
            axis=0,
        )

        X_list.append(X_i.astype(np.float32))

        rows.append({
            "index": i,
            "mass_group": label,
            "mass_1": network.params.mass_1,
            "mass_2": network.params.mass_2,
            "distance": network.params.distance,
            "chirp_mass": network.params.chirp_mass,
            "total_mass": network.params.total_mass,
            "network_snr": network.network_snr,
            "signal_network_duration": network.placement.signal_network_duration,
            "used_window_duration": network.windowed.metadata.used_window_duration,
            "full_network_duration": network.windowed.metadata.full_network_duration,
        })

    return np.stack(X_list, axis=0), pd.DataFrame(rows)


X_signal_fixed, fixed_meta_df = build_signal_only_segments(
    builder=builder,
    params_list=fixed_params,
    labels=fixed_labels,
    placement_policy="end_aligned",
)

print("X_signal_fixed:", X_signal_fixed.shape)
fixed_meta_df

## 4. Apply processing presets to controlled signal-only samples

The controlled signal-only samples currently have the final output length `config.length`.

The production pipeline normally processes a longer context segment and then crops back to `config.length`. Here, for a first controlled comparison, we use `output_mode="restore_length"` so that each processor can be applied directly to the final-length signal-only samples.

This isolates the effect of whitening, filtering and standardization on the same clean signals.

In [ ]:
from pycbc.types.timeseries import TimeSeries

def make_restore_length_preset(kwargs):
    """
    Convert a production-style preset into an audit preset that can process
    final-length signals directly.

    This is for controlled inspection only. Production uses crop_to_config
    with processing context.
    """
    out = dict(kwargs)
    out["output_mode"] = "restore_length"
    return out


AUDIT_PROCESSING_PRESETS = {
    name: make_restore_length_preset(kwargs)
    for name, kwargs in PROCESSING_PRESETS.items()
}

AUDIT_PROCESSING_PRESETS

In [ ]:
psds = {
    det: builder.noise_model.get_psd(det)
    for det in detector_names
}

for det, psd in psds.items():
    print(det, "len:", len(psd), "delta_f:", psd.delta_f)

In [ ]:
def array_to_timeseries(x, config, epoch=0.0):
    return TimeSeries(
        initial_array=np.asarray(x, dtype=np.float64),
        delta_t=config.delta_t,
        epoch=epoch,
    )


def process_array_network(
    X,
    processor,
    detector_names,
    config,
    psds=None,
):
    """
    Process an array with shape (n_samples, n_detectors, n_time).

    Returns
    -------
    X_processed : np.ndarray
        Shape (n_samples, n_detectors, config.length)
    """
    processed_samples = []

    for i in range(X.shape[0]):
        strains = {
            det: array_to_timeseries(
                X[i, det_idx],
                config=config,
                epoch=0.0,
            )
            for det_idx, det in enumerate(detector_names)
        }

        processed = processor.process_network(
            strains=strains,
            psds=psds,
        )

        Xi = np.stack(
            [np.asarray(processed[det]) for det in detector_names],
            axis=0,
        )

        processed_samples.append(Xi.astype(np.float32))

    return np.stack(processed_samples, axis=0)

In [ ]:
X_by_preset = {}

for name, kwargs in AUDIT_PROCESSING_PRESETS.items():
    print("Processing preset:", name)

    processor = SignalProcessor(
        config=config,
        **kwargs,
    )

    preset_psds = psds if processor.whitening_method == "psd" else None

    X_proc = process_array_network(
        X=X_signal_fixed,
        processor=processor,
        detector_names=detector_names,
        config=config,
        psds=preset_psds,
    )

    X_by_preset[name] = X_proc

    print("  shape:", X_proc.shape)
    print("  finite:", np.all(np.isfinite(X_proc)))
    print("  mean:", X_proc.mean())
    print("  std:", X_proc.std())
    print("  max_abs:", np.max(np.abs(X_proc)))
    print()

In [ ]:
summary_rows = []

for name, X_proc in X_by_preset.items():
    summary_rows.append({
        "preset": name,
        "shape": X_proc.shape,
        "finite": np.all(np.isfinite(X_proc)),
        "mean": float(X_proc.mean()),
        "std": float(X_proc.std()),
        "min": float(X_proc.min()),
        "max": float(X_proc.max()),
        "max_abs": float(np.max(np.abs(X_proc))),
        "rms": float(np.sqrt(np.mean(X_proc.astype(np.float64)**2))),
    })

processing_summary_df = pd.DataFrame(summary_rows)
processing_summary_df

## 5. Visual comparison

We now compare the same physical signals after each preprocessing preset.

The goal is not yet to pick the best preset, but to identify which transformations preserve or distort the visible mass-dependent morphology.

In [ ]:
def plot_processed_comparison_for_mass(
    X_by_preset,
    mass_index,
    mass_label,
    detector_idx=0,
    detector_name="H1",
    fs=4096.0,
    xlim=None,
):
    """
    Plot one detector for the same mass sample across processing presets.
    """
    n_presets = len(X_by_preset)

    fig, axes = plt.subplots(
        n_presets,
        1,
        figsize=(14, 2.4 * n_presets),
        sharex=True,
    )

    if n_presets == 1:
        axes = [axes]

    for ax, (preset_name, X_proc) in zip(axes, X_by_preset.items()):
        x = X_proc[mass_index, detector_idx]
        t = np.arange(len(x)) / fs

        ax.plot(t, x, alpha=0.9)
        ax.set_ylabel(detector_name)
        ax.set_title(preset_name)
        ax.grid(alpha=0.3)

        if xlim is not None:
            ax.set_xlim(*xlim)

    axes[-1].set_xlabel("Time [s]")

    fig.suptitle(
        f"Processing comparison | {mass_label} | detector {detector_name}",
        y=1.02,
    )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_processed_comparison_for_mass(
    X_by_preset=X_by_preset,
    mass_index=0,
    mass_label=20.0,
    detector_idx=0,
    detector_name="H1",
    fs=config.sampling_frequency,
    xlim=(30.0, 32.0),
)

In [ ]:
plot_processed_comparison_for_mass(
    X_by_preset=X_by_preset,
    mass_index=3,
    mass_label=80.0,
    detector_idx=0,
    detector_name="H1",
    fs=config.sampling_frequency,
    xlim=(30.0, 32.0),
)

In [ ]:
def normalized_energy_profiles(X, n_windows=128):
    """
    Compute normalized network energy profiles.

    X shape: (n_samples, n_detectors, n_time)
    returns shape: (n_samples, n_windows)
    """
    X64 = X.astype(np.float64)
    n, c, t = X64.shape

    usable = (t // n_windows) * n_windows
    X64 = X64[:, :, :usable]

    Xw = X64.reshape(n, c, n_windows, usable // n_windows)
    energy = np.sum(Xw**2, axis=(1, 3))

    total = energy.sum(axis=1, keepdims=True)

    if np.any(total <= 0):
        bad = np.where(total.reshape(-1) <= 0)[0]
        raise ValueError(f"Found zero-energy samples: {bad}")

    return energy / total

In [ ]:
def plot_energy_profiles_for_preset(
    X_proc,
    preset_name,
    mass_labels,
    n_windows=128,
):
    profiles = normalized_energy_profiles(X_proc, n_windows=n_windows)

    plt.figure(figsize=(10, 5))

    for i, mass in enumerate(mass_labels):
        plt.plot(
            profiles[i],
            label=f"{mass:.0f}+{mass:.0f}",
        )

    plt.xlabel("time window")
    plt.ylabel("normalized network energy")
    plt.title(f"Normalized network energy profile | {preset_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    return profiles

In [ ]:
profiles_by_preset = {}

for name, X_proc in X_by_preset.items():
    profiles_by_preset[name] = plot_energy_profiles_for_preset(
        X_proc=X_proc,
        preset_name=name,
        mass_labels=fixed_labels,
        n_windows=128,
    )

In [ ]:
active_rows = []

for preset_name, profiles in profiles_by_preset.items():
    for i, mass in enumerate(fixed_labels):
        p = profiles[i]

        active_rows.append({
            "preset": preset_name,
            "mass_group": mass,
            "active_1pct": int(np.sum(p > 0.01 * p.max())),
            "active_5pct": int(np.sum(p > 0.05 * p.max())),
            "peak_window": int(np.argmax(p)),
            "max_profile_energy": float(p.max()),
        })

active_by_preset_df = pd.DataFrame(active_rows)
active_by_preset_df

In [ ]:
active_pivot = active_by_preset_df.pivot_table(
    index="preset",
    columns="mass_group",
    values=["active_1pct", "active_5pct", "max_profile_energy"],
)

active_pivot

NOTES:

The current processing context is consistent with the estimated corrupted margins from whitening and FIR filtering. For the baseline whitening+bandpass preset, the estimated corrupted margin is ~0.375 s per side and the configured processing context is ~0.406 s per side.

In clean signal-only controls, raw and bandpass-only processing preserve the expected temporal spread difference between mass groups more clearly. PSD whitening compresses the normalized temporal energy profile, concentrating high-mass systems into one active window and reducing the 20+20 active support from ~3 to ~2 windows. Lowpass 512 Hz and 1024 Hz behave almost identically for the tested high-mass BBH cases.

Therefore, whitening+bandpass is physically and numerically defensible as a baseline, but it should be compared against bandpass-only and standardized whitening in noisy/training experiments before being treated as the final preprocessing choice.

## 6. Noisy controlled samples

We now add controlled Gaussian detector noise to the fixed signal-only samples.

The purpose is to test how different preprocessing presets behave when the signal is embedded in noise.

We compare only three candidate presets:

- `P1_bandpass_30_512`
- `P3_whiten_bandpass_30_512_current`
- `P4_whiten_bandpass_30_512_standardized`

Before adding noise, we rescale the clean signals to a common target network SNR. This removes the trivial amplitude shortcut where larger masses are easier to identify just because they have larger SNR at fixed distance.

In [ ]:
NOISY_PRESET_NAMES = [
    "P1_bandpass_30_512",
    "P3_whiten_bandpass_30_512_current",
    "P4_whiten_bandpass_30_512_standardized",
]

NOISY_PROCESSING_PRESETS = {
    name: AUDIT_PROCESSING_PRESETS[name]
    for name in NOISY_PRESET_NAMES
}

NOISY_PROCESSING_PRESETS

In [ ]:
def rescale_signal_network_to_target_snr(X, current_snr, target_snr):
    """
    Rescale signal-only strain arrays so that their network SNR would match target_snr.

    SNR is linear in waveform amplitude, so this is equivalent to multiplying
    the signal by target_snr / current_snr.
    """
    scale = target_snr / current_snr
    return X * scale, scale


target_snr = 20.0

X_signal_fixed_snr = []
snr_scale_rows = []

for i, mass in enumerate(fixed_labels):
    current_snr = fixed_meta_df.loc[i, "network_snr"]

    X_scaled, scale = rescale_signal_network_to_target_snr(
        X_signal_fixed[i],
        current_snr=current_snr,
        target_snr=target_snr,
    )

    X_signal_fixed_snr.append(X_scaled)

    snr_scale_rows.append({
        "mass_group": mass,
        "original_snr": current_snr,
        "target_snr": target_snr,
        "amplitude_scale": scale,
    })

X_signal_fixed_snr = np.stack(X_signal_fixed_snr, axis=0).astype(np.float32)

snr_scale_df = pd.DataFrame(snr_scale_rows)
snr_scale_df

In [ ]:
def generate_noisy_repeats_from_fixed_signals(
    X_signal_by_mass,
    mass_labels,
    builder,
    detector_names,
    n_repeats_per_mass=25,
    base_seed=5000,
):
    """
    Generate noisy samples by adding independent detector noise to fixed signal-only samples.

    Parameters
    ----------
    X_signal_by_mass:
        Array with shape (n_masses, n_detectors, n_time).
    mass_labels:
        Mass label for each clean signal.
    builder:
        DatasetBuilder with a configured NoiseModel.
    detector_names:
        Detector order.
    n_repeats_per_mass:
        Number of independent noise realizations per mass group.
    base_seed:
        Base random seed.

    Returns
    -------
    X_noisy:
        Shape (n_masses * n_repeats_per_mass, n_detectors, n_time).
    y_mass:
        Mass labels.
    repeat_df:
        Metadata table.
    """
    X_rows = []
    y_rows = []
    meta_rows = []

    sample_counter = 0

    for mass_idx, mass in enumerate(mass_labels):
        signal = X_signal_by_mass[mass_idx]

        for rep in range(n_repeats_per_mass):
            seed = base_seed + sample_counter

            noises = builder.noise_model.sample_network(
                detector_names=detector_names,
                seed=seed,
                length=config.length,
            )

            noise_arr = np.stack(
                [np.asarray(noises[det]) for det in detector_names],
                axis=0,
            ).astype(np.float32)

            noisy = signal + noise_arr

            X_rows.append(noisy.astype(np.float32))
            y_rows.append(float(mass))

            meta_rows.append({
                "sample_index": sample_counter,
                "mass_group": float(mass),
                "mass_idx": mass_idx,
                "repeat": rep,
                "seed": seed,
                "signal_rms": float(np.sqrt(np.mean(signal.astype(np.float64)**2))),
                "noise_rms": float(np.sqrt(np.mean(noise_arr.astype(np.float64)**2))),
                "noisy_rms": float(np.sqrt(np.mean(noisy.astype(np.float64)**2))),
            })

            sample_counter += 1

    X_noisy = np.stack(X_rows, axis=0)
    y_mass = np.asarray(y_rows, dtype=float)
    repeat_df = pd.DataFrame(meta_rows)

    return X_noisy, y_mass, repeat_df

In [ ]:
X_noisy_control, y_noisy_mass, noisy_meta_df = generate_noisy_repeats_from_fixed_signals(
    X_signal_by_mass=X_signal_fixed_snr,
    mass_labels=fixed_labels,
    builder=builder,
    detector_names=detector_names,
    n_repeats_per_mass=25,
    base_seed=5000,
)

print("X_noisy_control:", X_noisy_control.shape)
print("y_noisy_mass:", y_noisy_mass.shape)

noisy_meta_df.head()

In [ ]:
noisy_meta_df.groupby("mass_group")[
    ["signal_rms", "noise_rms", "noisy_rms"]
].describe()

In [ ]:
def plot_network_array_from_array(
    X,
    sample_idx,
    title="",
    fs=4096.0,
    detector_names=("H1", "L1", "V1"),
    xlim=None,
):
    x = X[sample_idx]
    t = np.arange(x.shape[-1]) / fs

    fig, axes = plt.subplots(
        x.shape[0],
        1,
        figsize=(14, 2.7 * x.shape[0]),
        sharex=True,
    )

    if x.shape[0] == 1:
        axes = [axes]

    for det_idx, det in enumerate(detector_names):
        axes[det_idx].plot(t, x[det_idx], alpha=0.9)
        axes[det_idx].set_ylabel(det)
        axes[det_idx].grid(alpha=0.3)

        if xlim is not None:
            axes[det_idx].set_xlim(*xlim)

    axes[-1].set_xlabel("Time [s]")
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
for mass in fixed_labels:
    idx = noisy_meta_df.query("mass_group == @mass").iloc[0]["sample_index"]
    idx = int(idx)

    plot_network_array_from_array(
        X_noisy_control,
        sample_idx=idx,
        title=f"Noisy controlled sample | {mass:.0f}+{mass:.0f} | target signal SNR={target_snr}",
        fs=config.sampling_frequency,
        detector_names=detector_names,
        xlim=(28.0, 32.0),
    )

In [ ]:
X_noisy_by_preset = {}

for name, kwargs in NOISY_PROCESSING_PRESETS.items():
    print("Processing noisy samples with:", name)

    processor = SignalProcessor(
        config=config,
        **kwargs,
    )

    preset_psds = psds if processor.whitening_method == "psd" else None

    X_proc = process_array_network(
        X=X_noisy_control,
        processor=processor,
        detector_names=detector_names,
        config=config,
        psds=preset_psds,
    )

    X_noisy_by_preset[name] = X_proc

    print("  shape:", X_proc.shape)
    print("  finite:", np.all(np.isfinite(X_proc)))
    print("  mean:", X_proc.mean())
    print("  std:", X_proc.std())
    print("  max_abs:", np.max(np.abs(X_proc)))
    print()

In [ ]:
noisy_processing_rows = []

for name, X_proc in X_noisy_by_preset.items():
    noisy_processing_rows.append({
        "preset": name,
        "shape": X_proc.shape,
        "finite": np.all(np.isfinite(X_proc)),
        "mean": float(X_proc.mean()),
        "std": float(X_proc.std()),
        "min": float(X_proc.min()),
        "max": float(X_proc.max()),
        "max_abs": float(np.max(np.abs(X_proc))),
        "rms": float(np.sqrt(np.mean(X_proc.astype(np.float64)**2))),
    })

noisy_processing_summary_df = pd.DataFrame(noisy_processing_rows)
noisy_processing_summary_df

In [ ]:
def plot_noisy_processed_comparison(
    X_by_preset,
    y_mass,
    target_mass,
    sample_within_mass=0,
    detector_idx=0,
    detector_name="H1",
    fs=4096.0,
    xlim=(28.0, 32.0),
):
    candidate_indices = np.where(y_mass == target_mass)[0]
    sample_idx = int(candidate_indices[sample_within_mass])

    fig, axes = plt.subplots(
        len(X_by_preset),
        1,
        figsize=(14, 2.6 * len(X_by_preset)),
        sharex=True,
    )

    if len(X_by_preset) == 1:
        axes = [axes]

    for ax, (preset_name, X_proc) in zip(axes, X_by_preset.items()):
        x = X_proc[sample_idx, detector_idx]
        t = np.arange(len(x)) / fs

        ax.plot(t, x, alpha=0.9)
        ax.set_ylabel(detector_name)
        ax.set_title(preset_name)
        ax.grid(alpha=0.3)

        if xlim is not None:
            ax.set_xlim(*xlim)

    axes[-1].set_xlabel("Time [s]")

    fig.suptitle(
        f"Noisy processed comparison | {target_mass:.0f}+{target_mass:.0f} | detector {detector_name}",
        y=1.02,
    )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_noisy_processed_comparison(
    X_by_preset=X_noisy_by_preset,
    y_mass=y_noisy_mass,
    target_mass=20.0,
    sample_within_mass=0,
    detector_idx=0,
    detector_name="H1",
    fs=config.sampling_frequency,
    xlim=(28.0, 32.0),
)

In [ ]:
plot_noisy_processed_comparison(
    X_by_preset=X_noisy_by_preset,
    y_mass=y_noisy_mass,
    target_mass=80.0,
    sample_within_mass=0,
    detector_idx=0,
    detector_name="H1",
    fs=config.sampling_frequency,
    xlim=(28.0, 32.0),
)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
def evaluate_noisy_mass_classifier(
    X,
    y,
    title,
    stride=16,
    test_size=0.25,
    seed=123,
):
    """
    Downsample processed time series and train a simple logistic regression classifier.
    """
    X_down = X[:, :, ::stride]
    X_feat = X_down.reshape(X_down.shape[0], -1)

    X_train, X_test, y_train, y_test = train_test_split(
        X_feat,
        y,
        test_size=test_size,
        random_state=seed,
        stratify=y,
    )

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, C=1.0),
    )

    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    print("=" * 80)
    print(title)
    print("=" * 80)
    print("X_feat:", X_feat.shape)
    print("X_train:", X_train.shape)
    print("X_test:", X_test.shape)
    print("accuracy:", accuracy_score(y_test, pred))
    print()
    print(classification_report(y_test, pred))
    print()
    print("confusion matrix:")
    print(confusion_matrix(y_test, pred))

    return clf, pred, y_test

In [ ]:
noisy_classifier_results = []

for name, X_proc in X_noisy_by_preset.items():
    clf, pred, y_test = evaluate_noisy_mass_classifier(
        X=X_proc,
        y=y_noisy_mass,
        title=f"Noisy controlled classification | {name}",
        stride=16,
    )

In [ ]:
noisy_profiles_by_preset = {}

for name, X_proc in X_noisy_by_preset.items():
    profiles = normalized_energy_profiles(X_proc, n_windows=128)
    noisy_profiles_by_preset[name] = profiles

    plt.figure(figsize=(10, 5))

    for mass in fixed_labels:
        p = profiles[y_noisy_mass == mass]
        mean_p = p.mean(axis=0)
        std_p = p.std(axis=0)

        x = np.arange(mean_p.shape[0])
        lower = np.maximum(mean_p - std_p, 0.0)
        upper = mean_p + std_p

        plt.plot(x, mean_p, label=f"{mass:.0f}+{mass:.0f}")
        plt.fill_between(x, lower, upper, alpha=0.15)

    plt.xlabel("time window")
    plt.ylabel("normalized network energy")
    plt.title(f"Noisy normalized energy profiles | {name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## Production-faithful audit

In [ ]:
def add_processing_context_to_array(X, config):
    """
    Add zero context before and after a final-length array.

    X shape: (n_samples, n_detectors, config.length)

    Returns
    -------
    X_context:
        Shape (n_samples, n_detectors, config.processing_length)
    """
    n, c, t = X.shape

    assert t == config.length

    left = config.processing_context_start_samples
    right = config.processing_context_end_samples

    X_context = np.zeros(
        (n, c, left + t + right),
        dtype=X.dtype,
    )

    X_context[:, :, left:left + t] = X

    assert X_context.shape[-1] == config.processing_length

    return X_context

In [ ]:
def process_array_network_with_context(
    X,
    processor,
    detector_names,
    config,
    psds=None,
):
    """
    Process final-length arrays using production-like context.

    Input X has shape (n_samples, n_detectors, config.length).
    Internally, we pad context and use output_mode='crop_to_config'.
    """
    X_context = add_processing_context_to_array(X, config)

    processed_samples = []

    for i in range(X_context.shape[0]):
        strains = {
            det: array_to_timeseries(
                X_context[i, det_idx],
                config=config,
                epoch=0.0,
            )
            for det_idx, det in enumerate(detector_names)
        }

        processed = processor.process_network(
            strains=strains,
            psds=psds,
        )

        Xi = np.stack(
            [np.asarray(processed[det]) for det in detector_names],
            axis=0,
        )

        processed_samples.append(Xi.astype(np.float32))

    return np.stack(processed_samples, axis=0)

In [ ]:
PRODUCTION_TEST_PRESETS = {
    name: PROCESSING_PRESETS[name]
    for name in [
        "P1_bandpass_30_512",
        "P3_whiten_bandpass_30_512_current",
        "P4_whiten_bandpass_30_512_standardized",
    ]
}

In [ ]:
X_noisy_by_preset_prodlike = {}

psds_processing = {
    det: builder.noise_model.get_psd(
        det,
        length=config.processing_length,
    )
    for det in detector_names
}

for name, kwargs in PRODUCTION_TEST_PRESETS.items():
    print("Production-like processing:", name)

    processor = SignalProcessor(
        config=config,
        **kwargs,
    )

    preset_psds = psds_processing if processor.whitening_method == "psd" else None

    X_proc = process_array_network_with_context(
        X=X_noisy_control,
        processor=processor,
        detector_names=detector_names,
        config=config,
        psds=preset_psds,
    )

    X_noisy_by_preset_prodlike[name] = X_proc

    print("  shape:", X_proc.shape)
    print("  finite:", np.all(np.isfinite(X_proc)))
    print("  mean:", X_proc.mean())
    print("  std:", X_proc.std())
    print("  max_abs:", np.max(np.abs(X_proc)))
    print()

In [ ]:
for name, X_proc in X_noisy_by_preset_prodlike.items():
    clf, pred, y_test = evaluate_noisy_mass_classifier(
        X=X_proc,
        y=y_noisy_mass,
        title=f"Production-like noisy controlled classification | {name}",
        stride=16,
    )

Decisión provisional:

P1_bandpass_30_512:
  candidato principal alternativo.
  Conserva mejor la separabilidad de masa en controles noisy.

P3_whiten_bandpass_30_512_current:
  baseline actual.
  Metodológicamente defendible, pero aparentemente menos informativo para este test.

P4_whiten_bandpass_30_512_standardized:
  candidato secundario.
  Sólo merece seguir si al entrenar CNN/foundation model mejora estabilidad u optimización.

P5_lowpass1024:
  fuera por ahora.
  No vimos mejora clara frente a P3.

P0/P2:
  controles, no presets finales.

NOTES:

In production-like controlled noisy tests, bandpass-only preprocessing achieved higher mass-group separability than the current whitening+bandpass baseline. This suggests that PSD whitening, while physically well-motivated and useful for noise conditioning, may reduce the accessibility of mass-dependent temporal morphology for this task. However, the current test uses repeated noise realizations of fixed clean signals, so it is not sufficient for final selection. The next test must introduce extrinsic variability and evaluate generalization across unseen clean signals.

## 8. Four-second processing audit for CNN dataset

In [ ]:
config_4s = SimulationConfig(
    simulation_regime="BBH",
    waveform_family="IMR",

    sampling_frequency=4096.0,
    duration=4.0,

    low_frequency_cutoff=30.0,
    waveform_approximant="SEOBNRv4_opt",

    target_network_snr_range=(10.0, 25.0),
    snr_relative_tolerance=0.05,
    snr_on_truncated_signal=True,

    truncation_policy="keep_last_segment",
    required_final_duration=1.0,

    safe_margin_start=0.0,
    safe_margin_end=0.0,

    processing_context_start_samples=1664,
    processing_context_end_samples=1664,
)

print("duration:", config_4s.duration)
print("sampling_frequency:", config_4s.sampling_frequency)
print("length:", config_4s.length)
print("processing_length:", config_4s.processing_length)
print("processing context start [s]:", config_4s.processing_context_start_seconds)
print("processing context end [s]:", config_4s.processing_context_end_seconds)

In [ ]:
PROCESSING_PRESETS_4S = {
    "P1_bandpass_30_512": {
        "whitening_method": "none",
        "apply_highpass": True,
        "apply_lowpass": True,
        "apply_standardization": False,
        "output_mode": "crop_to_config",
        "highpass_frequency": 30.0,
        "lowpass_frequency": 512.0,
        "fir_order": 256,
        "fir_beta": 5.0,
        "remove_corrupted": True,
    },

    "P3_whiten_bandpass_30_512_current": {
        "whitening_method": "psd",
        "apply_highpass": True,
        "apply_lowpass": True,
        "apply_standardization": False,
        "output_mode": "crop_to_config",
        "whitening_low_frequency_cutoff": 30.0,
        "whitening_max_filter_duration": 0.5,
        "whitening_trunc_method": "hann",
        "highpass_frequency": 30.0,
        "lowpass_frequency": 512.0,
        "fir_order": 256,
        "fir_beta": 5.0,
        "remove_corrupted": True,
    },

    "P4_whiten_bandpass_30_512_standardized": {
        "whitening_method": "psd",
        "apply_highpass": True,
        "apply_lowpass": True,
        "apply_standardization": True,
        "output_mode": "crop_to_config",
        "whitening_low_frequency_cutoff": 30.0,
        "whitening_max_filter_duration": 0.5,
        "whitening_trunc_method": "hann",
        "highpass_frequency": 30.0,
        "lowpass_frequency": 512.0,
        "fir_order": 256,
        "fir_beta": 5.0,
        "remove_corrupted": True,
    },

    "P6_bandpass_30_1024": {
        "whitening_method": "none",
        "apply_highpass": True,
        "apply_lowpass": True,
        "apply_standardization": False,
        "output_mode": "crop_to_config",
        "highpass_frequency": 30.0,
        "lowpass_frequency": 1024.0,
        "fir_order": 256,
        "fir_beta": 5.0,
        "remove_corrupted": True,
    },
}

In [ ]:
preset_rows_4s = []

for name, kwargs in PROCESSING_PRESETS_4S.items():
    processor = SignalProcessor(
        config=config_4s,
        **kwargs,
    )

    meta = processor.metadata()

    preset_rows_4s.append({
        "preset": name,
        "processing_preset_name": meta["processing_preset"],
        "whitening_method": meta["whitening_method"],
        "apply_highpass": meta["apply_highpass"],
        "apply_lowpass": meta["apply_lowpass"],
        "apply_standardization": meta["apply_standardization"],
        "highpass_frequency": meta["highpass_frequency"],
        "lowpass_frequency": meta["lowpass_frequency"],
        "corrupted_margin_seconds_per_side": meta["corrupted_margin_seconds_per_side"],
        "recommended_safe_margin_start": meta["recommended_safe_margin_start"],
        "recommended_safe_margin_end": meta["recommended_safe_margin_end"],
        "usable_duration_after_processing_margins": meta["usable_duration_after_processing_margins"],
        "output_duration": meta["output_duration"],
        "processing_input_duration": meta["processing_input_duration"],
    })

preset_df_4s = pd.DataFrame(preset_rows_4s)
preset_df_4s

In [ ]:
def make_fixed_params_4s(mass_1, mass_2=None, distance=1000.0):
    if mass_2 is None:
        mass_2 = mass_1

    return CBCParameters(
        mass_1=float(mass_1),
        mass_2=float(mass_2),
        distance=float(distance),
        inclination=0.7,
        ra=1.0,
        dec=0.5,
        spin_1z=0.0,
        spin_2z=0.0,
        polarization_angle=0.0,
    )

In [ ]:
mass_cases_4s = [
    ("5+5", make_fixed_params_4s(5, 5)),
    ("10+10", make_fixed_params_4s(10, 10)),
    ("20+20", make_fixed_params_4s(20, 20)),
    ("40+40", make_fixed_params_4s(40, 40)),
    ("80+80", make_fixed_params_4s(80, 80)),
    ("90+90", make_fixed_params_4s(90, 90)),
    ("5+20", make_fixed_params_4s(20, 5)),
    ("10+60", make_fixed_params_4s(60, 10)),
]

In [ ]:
builder_4s = DatasetBuilder.from_config(
    config=config_4s,
    detector_names=detector_names,
    signal_processor_kwargs=PROCESSING_PRESETS_4S["P3_whiten_bandpass_30_512_current"],
    label_transformer_kwargs={},
    parameter_sampler_kwargs={
        "regime": "BBH",
        "fixed": {},
    },
    rng=np.random.default_rng(1234),
)

In [ ]:
case_names_4s = [name for name, _ in mass_cases_4s]
params_4s = [p for _, p in mass_cases_4s]

X_signal_4s, meta_4s = build_signal_only_segments(
    builder=builder_4s,
    params_list=params_4s,
    labels=np.asarray(case_names_4s),
    placement_policy="end_aligned",
)

print("X_signal_4s:", X_signal_4s.shape)
meta_4s

In [ ]:
labels_4s = np.arange(len(case_names_4s))

In [ ]:
meta_4s[
    [
        "mass_group",
        "mass_1",
        "mass_2",
        "chirp_mass",
        "total_mass",
        "network_snr",
        "signal_network_duration",
        "used_window_duration",
        "full_network_duration",
    ]
]

In [ ]:
def add_processing_context_to_array_with_config(X, config):
    n, c, t = X.shape

    assert t == config.length

    left = config.processing_context_start_samples
    right = config.processing_context_end_samples

    X_context = np.zeros(
        (n, c, left + t + right),
        dtype=X.dtype,
    )

    X_context[:, :, left:left + t] = X

    assert X_context.shape[-1] == config.processing_length

    return X_context


def process_array_network_with_context_config(
    X,
    processor,
    detector_names,
    config,
    psds=None,
):
    X_context = add_processing_context_to_array_with_config(X, config)

    processed_samples = []

    for i in range(X_context.shape[0]):
        strains = {
            det: TimeSeries(
                initial_array=np.asarray(X_context[i, det_idx], dtype=np.float64),
                delta_t=config.delta_t,
                epoch=0.0,
            )
            for det_idx, det in enumerate(detector_names)
        }

        processed = processor.process_network(
            strains=strains,
            psds=psds,
        )

        Xi = np.stack(
            [np.asarray(processed[det]) for det in detector_names],
            axis=0,
        )

        processed_samples.append(Xi.astype(np.float32))

    return np.stack(processed_samples, axis=0)

In [ ]:
psds_4s = {
    det: builder_4s.noise_model.get_psd(det)
    for det in detector_names
}

for det, psd in psds_4s.items():
    print(det, "len:", len(psd), "delta_f:", psd.delta_f)

In [ ]:
print("config_4s.length:", config_4s.length)
print("config_4s.processing_length:", config_4s.processing_length)
print("final flength:", config_4s.flength)
print("processing flength:", config_4s.processing_flength)
print("final delta_f:", config_4s.delta_f)
print("processing delta_f:", config_4s.processing_delta_f)

In [ ]:
from pycbc.psd import aLIGOZeroDetHighPower

def make_processing_psds(config, detector_names):
    psds = {}

    for det in detector_names:
        psds[det] = aLIGOZeroDetHighPower(
            length=config.processing_flength,
            delta_f=config.processing_delta_f,
            low_freq_cutoff=config.low_frequency_cutoff,
        )

    return psds


psds_4s_processing = {
    det: builder_4s.noise_model.get_psd(
        det,
        length=config_4s.processing_length,
    )
    for det in detector_names
}

for det, psd in psds_4s_processing.items():
    print(det, "len:", len(psd), "delta_f:", psd.delta_f)

In [ ]:
X_4s_by_preset = {}

for name, kwargs in PROCESSING_PRESETS_4S.items():
    print("4s production-like processing:", name)

    processor = SignalProcessor(
        config=config_4s,
        **kwargs,
    )

    preset_psds = psds_4s_processing if processor.whitening_method == "psd" else None

    X_proc = process_array_network_with_context_config(
        X=X_signal_4s,
        processor=processor,
        detector_names=detector_names,
        config=config_4s,
        psds=preset_psds,
    )

    X_4s_by_preset[name] = X_proc

    print("  shape:", X_proc.shape)
    print("  finite:", np.all(np.isfinite(X_proc)))
    print("  mean:", X_proc.mean())
    print("  std:", X_proc.std())
    print("  max_abs:", np.max(np.abs(X_proc)))
    print()

In [ ]:
for case_idx, case_name in enumerate(case_names_4s):
    if case_name not in ["5+5", "10+10", "80+80", "5+20", "90+90"]:
        continue

    plot_processed_comparison_for_mass(
        X_by_preset=X_4s_by_preset,
        mass_index=case_idx,
        mass_label=case_name,  # si la función asume float, cambia título dentro
        detector_idx=0,
        detector_name="H1",
        fs=config_4s.sampling_frequency,
        xlim=(0.0, 4.0),
    )

In [ ]:
profiles_4s_by_preset = {}

for name, X_proc in X_4s_by_preset.items():
    profiles = normalized_energy_profiles(X_proc, n_windows=64)
    profiles_4s_by_preset[name] = profiles

    plt.figure(figsize=(10, 5))

    for i, case_name in enumerate(case_names_4s):
        plt.plot(profiles[i], label=case_name)

    plt.xlabel("time window")
    plt.ylabel("normalized network energy")
    plt.title(f"4 s normalized network energy profile | {name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

El procesado actual P3 sigue siendo técnicamente defendible: el contexto cubre los márgenes corruptos y el whitening con PSD es coherente con ruido coloreado. Sin embargo, las auditorías muestran que P1 bandpass-only conserva mejor la morfología temporal visible y que P3/P4 reponderan/concentran fuertemente la señal. Por tanto, P1 debe pasar a la siguiente ronda como candidato serio para entrenamiento, junto con P3 como baseline actual y P4 como variante normalizada.

In [ ]:
active_4s_rows = []

for preset_name, profiles in profiles_4s_by_preset.items():
    for i, case_name in enumerate(case_names_4s):
        p = profiles[i]

        active_4s_rows.append({
            "preset": preset_name,
            "case": case_name,
            "active_1pct": int(np.sum(p > 0.01 * p.max())),
            "active_5pct": int(np.sum(p > 0.05 * p.max())),
            "peak_window": int(np.argmax(p)),
            "max_profile_energy": float(p.max()),
        })

active_4s_df = pd.DataFrame(active_4s_rows)
active_4s_df

In [ ]:
active_4s_pivot = active_4s_df.pivot_table(
    index="preset",
    columns="case",
    values=["active_1pct", "active_5pct", "max_profile_energy"],
)

active_4s_pivot

Esto indica que whitening + bandpass hace que la energía útil quede más concentrada. No necesariamente destruye la señal, pero sí cambia la representación de forma fuerte. Para una CNN puede ser bueno o malo; hay que probarlo. Pero ya no se puede asumir que P3 sea neutral.

El SignalProcessor hace whitening dividiendo por la raíz de la PSD condicionada y después aplica highpass/lowpass FIR si están activos; por tanto es esperable que repondere la señal en frecuencia y cambie su forma temporal aparente.

lowpass=1024 Hz no aporta una diferencia clara frente a lowpass=512 Hz en esta auditoría signal-only 4 s.

Final decision:

P1_bandpass_30_512
  Mantener. Candidato serio. Conserva mejor estructura temporal, especialmente en masas bajas.

P3_whiten_bandpass_30_512_current
  Mantener. Baseline actual, físicamente defendible para ruido coloreado/PSD.

P4_whiten_bandpass_30_512_standardized
  Mantener sólo como variante de entrenamiento/optimización.

P6_bandpass_30_1024
  Pausar/descartar por ahora. No aporta diferencia clara frente a P1.

P0/P2
  Sólo controles, no candidatos finales.

## 9. Four-second noisy audit with variable extrinsics

In [ ]:
from src.sampling import PriorConfig, ParameterSampler

def sample_params_fixed_masses(
    mass_1,
    mass_2,
    n_samples,
    seed,
    distance_range=(500.0, 5000.0),
):
    prior = PriorConfig.bbh(
        fixed_parameters={
            "mass_1": float(mass_1),
            "mass_2": float(mass_2),
            "spin_1z": 0.0,
            "spin_2z": 0.0,
        }
    )

    prior = PriorConfig(
        regime=prior.regime,
        component_mass_range=prior.component_mass_range,
        distance_range=distance_range,
        spin_1z_range=prior.spin_1z_range,
        spin_2z_range=prior.spin_2z_range,
        fixed_parameters=prior.fixed_parameters,
    )

    sampler = ParameterSampler(
        prior_config=prior,
        rng=np.random.default_rng(seed),
    )

    return sampler.sample_many(n_samples)

In [ ]:
case_specs_4s = [
    ("5+5", 5.0, 5.0),
    ("10+10", 10.0, 10.0),
    ("20+20", 20.0, 20.0),
    ("40+40", 40.0, 40.0),
    ("80+80", 80.0, 80.0),
    ("90+90", 90.0, 90.0),
    ("5+20", 20.0, 5.0),
    ("10+60", 60.0, 10.0),
]

n_base_per_case = 20

params_var_4s = []
case_labels_var_4s = []
base_ids_var_4s = []

base_counter = 0

for case_idx, (case_name, m1, m2) in enumerate(case_specs_4s):
    params_case = sample_params_fixed_masses(
        mass_1=m1,
        mass_2=m2,
        n_samples=n_base_per_case,
        seed=7000 + case_idx,
    )

    for p in params_case:
        params_var_4s.append(p)
        case_labels_var_4s.append(case_name)
        base_ids_var_4s.append(base_counter)
        base_counter += 1

case_labels_var_4s = np.asarray(case_labels_var_4s)
base_ids_var_4s = np.asarray(base_ids_var_4s)

print("n base signals:", len(params_var_4s))
pd.Series(case_labels_var_4s).value_counts().sort_index()

In [ ]:
X_signal_var_4s, meta_var_4s = build_signal_only_segments(
    builder=builder_4s,
    params_list=params_var_4s,
    labels=case_labels_var_4s,
    placement_policy="random_contained",
)

print("X_signal_var_4s:", X_signal_var_4s.shape)
meta_var_4s.head()

In [ ]:
meta_var_4s["case"] = case_labels_var_4s
meta_var_4s["base_id"] = base_ids_var_4s

meta_var_4s.groupby("case")[
    [
        "signal_network_duration",
        "used_window_duration",
        "full_network_duration",
        "network_snr",
    ]
].describe()

In [ ]:
target_snr_4s = 20.0

X_signal_var_4s_snr = []

for i in range(X_signal_var_4s.shape[0]):
    current_snr = float(meta_var_4s.loc[i, "network_snr"])
    scale = target_snr_4s / current_snr
    X_signal_var_4s_snr.append(X_signal_var_4s[i] * scale)

X_signal_var_4s_snr = np.stack(X_signal_var_4s_snr, axis=0).astype(np.float32)

print(X_signal_var_4s_snr.shape)

In [ ]:
def add_noise_to_signal_collection(
    X_signal,
    builder,
    detector_names,
    config,
    base_seed=9000,
):
    X_rows = []
    seeds = []

    for i in range(X_signal.shape[0]):
        seed = base_seed + i

        noises = builder.noise_model.sample_network(
            detector_names=detector_names,
            seed=seed,
            length=config.length,
        )

        noise_arr = np.stack(
            [np.asarray(noises[det]) for det in detector_names],
            axis=0,
        ).astype(np.float32)

        X_rows.append((X_signal[i] + noise_arr).astype(np.float32))
        seeds.append(seed)

    return np.stack(X_rows, axis=0), np.asarray(seeds)


X_noisy_var_4s, noise_seeds_4s = add_noise_to_signal_collection(
    X_signal=X_signal_var_4s_snr,
    builder=builder_4s,
    detector_names=detector_names,
    config=config_4s,
    base_seed=9000,
)

print("X_noisy_var_4s:", X_noisy_var_4s.shape)

In [ ]:
psds_4s_processing = {
    det: builder_4s.noise_model.get_psd(
        det,
        length=config_4s.processing_length,
    )
    for det in detector_names
}

In [ ]:
CANDIDATE_PRESETS_4S = {
    name: PROCESSING_PRESETS_4S[name]
    for name in [
        "P1_bandpass_30_512",
        "P3_whiten_bandpass_30_512_current",
        "P4_whiten_bandpass_30_512_standardized",
    ]
}

X_noisy_var_4s_by_preset = {}

for name, kwargs in CANDIDATE_PRESETS_4S.items():
    print("Processing:", name)

    processor = SignalProcessor(
        config=config_4s,
        **kwargs,
    )

    preset_psds = psds_4s_processing if processor.whitening_method == "psd" else None

    X_proc = process_array_network_with_context_config(
        X=X_noisy_var_4s,
        processor=processor,
        detector_names=detector_names,
        config=config_4s,
        psds=preset_psds,
    )

    X_noisy_var_4s_by_preset[name] = X_proc

    print("  shape:", X_proc.shape)
    print("  finite:", np.all(np.isfinite(X_proc)))
    print("  mean/std:", X_proc.mean(), X_proc.std())
    print()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
def evaluate_case_classifier_4s(
    X,
    y_case,
    title,
    stride=4,
    test_size=0.25,
    seed=123,
):
    X_down = X[:, :, ::stride]
    X_feat = X_down.reshape(X_down.shape[0], -1)

    X_train, X_test, y_train, y_test = train_test_split(
        X_feat,
        y_case,
        test_size=test_size,
        random_state=seed,
        stratify=y_case,
    )

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, C=1.0),
    )

    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    print("=" * 80)
    print(title)
    print("=" * 80)
    print("X_feat:", X_feat.shape)
    print("accuracy:", accuracy_score(y_test, pred))
    print()
    print(classification_report(y_test, pred))
    print()
    print("confusion matrix:")
    print(confusion_matrix(y_test, pred))

    return clf, pred, y_test

In [ ]:
for name, X_proc in X_noisy_var_4s_by_preset.items():
    evaluate_case_classifier_4s(
        X=X_proc,
        y_case=case_labels_var_4s,
        title=f"4 s noisy variable-extrinsics case classification | {name}",
        stride=4,
    )

## Processing audit conclusion

The audit suggests that P1, P3 and P4 should be retained for CNN benchmarking.

P1 preserves temporal morphology more clearly in signal-only controls, especially for low-mass systems. P3 remains the current physically motivated whitening baseline. P4 is retained as a standardized variant of P3, mainly to test whether input scale improves CNN optimization.

P6 is paused because lowpass 1024 Hz did not show a meaningful difference from lowpass 512 Hz in the tested cases.